In [1]:
%%html
<script>
  function code_toggle() {
    if (code_shown){
      $('div.input').hide('500');
      $('#toggleButton').val('Show Code')
    } else {
      $('div.input').show('500');
      $('#toggleButton').val('Hide Code')
    }
    code_shown = !code_shown
  }

  $( document ).ready(function(){
    code_shown=false;
    $('div.input').hide()
  });
</script>
<form action="javascript:code_toggle()"><input type="submit" id="toggleButton" value="Show Code"></form>
<style>
.rendered_html td {
    font-size: xx-large;
    text-align: left; !important
}
.rendered_html th {
    font-size: xx-large;
    text-align: left; !important
}
</style>

In [2]:
%%capture
import sys
sys.path.append("..")
import statnlpbook.util as util
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10.0, 6.0)

<!---
Latex Macros
-->
$$
\newcommand{\Xs}{\mathcal{X}}
\newcommand{\Ys}{\mathcal{Y}}
\newcommand{\y}{\mathbf{y}}
\newcommand{\balpha}{\boldsymbol{\alpha}}
\newcommand{\bbeta}{\boldsymbol{\beta}}
\newcommand{\aligns}{\mathbf{a}}
\newcommand{\align}{a}
\newcommand{\source}{\mathbf{s}}
\newcommand{\target}{\mathbf{t}}
\newcommand{\ssource}{s}
\newcommand{\starget}{t}
\newcommand{\repr}{\mathbf{f}}
\newcommand{\repry}{\mathbf{g}}
\newcommand{\x}{\mathbf{x}}
\newcommand{\prob}{p}
\newcommand{\bar}{\,|\,}
\newcommand{\vocab}{V}
\newcommand{\params}{\boldsymbol{\theta}}
\newcommand{\param}{\theta}
\DeclareMathOperator{\perplexity}{PP}
\DeclareMathOperator{\argmax}{argmax}
\DeclareMathOperator{\argmin}{argmin}
\newcommand{\train}{\mathcal{D}}
\newcommand{\counts}[2]{\#_{#1}(#2) }
\newcommand{\length}[1]{\text{length}(#1) }
\newcommand{\indi}{\mathbb{I}}
$$

In [3]:
%load_ext tikzmagic

# Please complete the course evaluation

### [evaluering.ku.dk](https://evaluering.ku.dk)

Deadline: **November 1**

# Machine Translation

+ Challenges
+ History
+ Statistical MT
+ Neural MT
+ Evaluation


## Learning objectives

By the end of this lecture, you should be able to:

- explain why linguistic divergences make translation difficult and compare rule-based, statistical and neural approaches;
- formulate machine translation as structured prediction and describe autoregressive sequence-to-sequence decoding with attention; and
- compare human evaluation with BLEU, chrF and BERTScore, including their limitations.

## Languages are hard (even for humans)!

<center style="padding-top:3em;">
  <img src="mt_figures/whatever.jpg" />
    
  <span style="font-size:50%;">(Source: <a href="https://www.flickr.com/photos/98991392@N00/8729849093/sizes/z/in/pool-47169589@N00/">Flickr</a>)</span>
</center>

[随便](https://translate.google.com/#view=home&op=translate&sl=zh-CN&tl=en&text=%E9%9A%8F%E4%BE%BF)

## Automatic machine translation is hard!

<center style="padding-top:3em;">
<img src="../chapters/mt_figures/avocado.png" width="100%"/>
</center>

[J'ai besoin d'un avocat pour mon procés de guacamole.](https://translate.google.com/?sl=fr&tl=en&text=J%27ai%20besoin%20d%27un%20avocat%20pour%20mon%20proc%C3%A9s%20de%20guacamole.&op=translate)

[guacamole lawsuit](https://www.latimes.com/archives/la-xpm-2006-dec-10-fi-letters10.2-story.html)

Many things could go wrong.

<center><img src="../img/quiz_time.png"></center>

# [tinyurl.com/diku-nlp-mt](https://tinyurl.com/diku-nlp-mt)

([Responses](https://docs.google.com/forms/d/10UUpQqlduvYuz7-SXSvzMRDAs5qxdliKJmgnkWiHNK8/edit#responses))


## Challenges

Divergences between languages include:

* Word order
* Grammatical (morphological) marking (e.g., gender)
* Division of concept space (e.g., English "wall" vs. German "Mauer"/"Wand")

Addressing them requires resolving ambiguities:

* Word sense (e.g., "bass")
* Attributes with grammatical marking (e.g., formality in Japanese)
* Reference in pro-drop contexts (e.g., in Mandarin Chinese)

## History

<center>
   <img src="https://cdn-media-1.freecodecamp.org/images/l6Av1jJRcgtfL5n9vEvrspxpEJJalwHAusRx" width="80%" />
</center>
(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

### Direct Machine Translation

Just use a **dictionary** and translate word-by-word.

<center>
   <img src="https://cdn-media-1.freecodecamp.org/images/gfF1OTGY1E6QQBDW6l6kh1Daa4iingQmt86V" width="100%" />
</center>
(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

### Transfer-based Machine Translation

Add **rules** to inflect/join/split words based on source and target syntax.

<center>
   <img src="https://cdn-media-1.freecodecamp.org/images/AfcjFhnoMFmb8koHf42YQeE6WfKdFNjqpkGL" width="50%" />
</center>
(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

## Interlingua

The ideal: transform to and from a language-independent representation.

<center>
   <img src="https://cdn-media-1.freecodecamp.org/images/vEPJYMmjDV0LLXIy07LksIiOsecXdlHcs8nI" width="80%" />
</center>
(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

Too hard to achieve with rules!

### Example-based Machine Translation (EBMT)

Retrieve a similar example from a translation database, and make adjustments as necessary.

<center>
   <img src="https://cdn-media-1.freecodecamp.org/images/H-VtzpHN02SMIhwYjqmn04Uyd-nGWLwLmBwW" width="100%" />
</center>
(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

# Statistical Machine Translation (SMT)

## IBM Translation Models

In the late 80s and early 90s, IBM researchers revolutionised MT using statistical approaches instead of rules.

<img src="https://cdn-media-1.freecodecamp.org/images/1MJctJSHzUaYpUNIhvQdQtz4RKFq06nN7FJ9"/>
(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

### IBM Model 1

Simple word-by-word translation, but with **statistical** dictionaries.

<center>
   <img src="https://cdn-media-1.freecodecamp.org/images/4dBTKxFLuXkmeALMuxILitzmBd0zVOQVrhnP" width="80%" />
</center>

(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

<center>
   <img src="https://cdn-media-1.freecodecamp.org/images/uF96He1UZaLMkC1TEuBGzoHAhw3PAvF8mgam" width="50%" />
</center>

(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

### IBM Model 2
Statistical translation and **reordering** model.

<center>
   <img src="https://cdn-media-1.freecodecamp.org/images/AWurqrK2Zag9dIZSgYVGZCFxklsrZot7WLZ2" width="60%" />
</center>

(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

### IBM Model 3
Allows inserting new words.

<center>
   <img src="https://cdn-media-1.freecodecamp.org/images/aPxpW2ssFd2wDio9C51Zbb0sIdXLBAV8DoYP" width="50%" />
</center>

(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

## Word-based SMT

Decompose source and target to words, with statistical **alignment**.

<center>
   <img src="https://cdn-media-1.freecodecamp.org/images/jG95Sgc2W4VJbwi4LFlJeMHnjLZbdGydCCzI" width="100%" />
</center>


(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

## Phrase-based SMT

Decompose source and target to **phrases** and look them up in phrase tables.

<center>
   <img src="https://cdn-media-1.freecodecamp.org/images/lGJNqYGZOJMjs23F8-xf6E4buXptvm2IBzjg" width="70%" />
</center>


(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

## MT as Structured Prediction

**Model** \\(p(\target,\source)\\): how likely the target \\(\target\\) is to be a translation of source $\source$.

$$p(\textrm{I like music}, \textrm{音楽 が 好き}) \gg p(\textrm{I like persimmons}, \textrm{音楽 が 好き})$$

* How is the scoring function defined (**modeling**)?
$$\prob(\target,\source)\approx s_\params(\target,\source)$$

* How are the parameters \\(\params\\) learned (**training**)?

$$
\argmax_\params \prod_{(\target,\source) \in \train} s_\params(\target, \source)
$$

* How is translation \\(\argmax\\) found (**decoding**)?

$$\argmax_\target s_\params(\target,\source)$$

### Training
Learn the parameters \\(\params\\) from data.

<center>
   <img src="https://cdn-media-1.freecodecamp.org/images/5qBpooShbY6xVngSotA6KINHyHP7NKeTJryb" width="80%" />
</center>

(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

## Generative Models 
Estimate $\prob(\target,\source)$: how is the $(\target,\source)$ data **generated**?

### Noisy Channel

* Imagine a message $\target$ is sent through a noisy channel and $\source$ is received at the end.
* Can we recover what was $\target$?
* **Language model** $\prob(\target)$: does the target $\target$ look like real language?
* **Translation model**: $\prob(\source|\target)$: does the source $\source$ match the target $\target$?

This defines a **joint** distribution

$$\prob(\target,\source) = \prob(\target) \prob(\source|\target)$$

# Neural Machine Translation (NMT)

Model $s_\params(\target,\source)$ directly using a neural network.

<center>
   <img src="https://cdn-media-1.freecodecamp.org/images/DD6GvRmtxZGhC9toN1CHXUBWLhUXLpJSiJF5" width="70%" />
</center>


(Source: <a href="https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/">freeCodeCamp</a>)

## Sequence-to-sequence models (seq2seq)

Encoder–decoder architecture first "read" the input sequence and then generate an output sequence
([Sutskever et al., 2014](https://papers.nips.cc/paper/5346-sequence-to-sequence-learning-with-neural-networks.pdf), [Cho et al., 2014](https://arxiv.org/abs/1406.1078)).

&nbsp;

<center>
    <img src="mt_figures/encdec.svg" width="70%" />
</center>

(Examples are Basque–English)

### We can use RNNs for that!

Example architecture:
* Encoder: word embedding layer + Bi-LSTM to capture contextual information
* Decoder: Uni-directional LSTM (because we need to *decode* word by word) + softmax layer on top

<center>
    <img src="mt_figures/encdec_rnn1.svg" width="70%" />
</center>

The end-of-sequence symbol `</S>` is necessary to know when to start and stop generating.

### But something's missing (again)...

&nbsp;

<center>
    <img src="mt_figures/encdec_rnn1.svg" width="70%" />
</center>

Output words **depend on each other**!

## Autoregressive MT

At each step, feed the predicted word to the decoder as input for predicting the next word.

<center>
    <img src="mt_figures/encdec_rnn2.svg" width="70%" />
</center>

### Training

+ Loss function: negative log-likelihood
+ **Teacher forcing:**  always feed the ground truth into the decoder.

*Alternative:*

+ **Scheduled sampling:** with a certain probability, use model predictions instead.


### Decoding

+ Greedy decoding:
    + Always pick the **most likely word** (according to the softmax)
    + Continue generating more words **until the `</S>` symbol is predicted**

+ Beam search:
    * Keep a **beam of up to $k$ partial target sequences**.
    * Extend each sequence with possible next **target tokens**, conditioned on the source and that sequence's prefix.
    * Keep the best-scoring extensions by cumulative log-probability (often with length normalization).
    * Do not extend hypotheses ending in `</S>`. Stop when all beam entries are complete or a maximum length is reached.

## More problems with our approach

&nbsp;

<center>
    <img src="mt_figures/encdec_rnn3.svg" width="70%" />
</center>

### Word alignment

In rule-based and statistical MT, word alignments are crucial.

<center style="padding: 1em 0;">
    <img src="mt_figures/align.svg" width="20%" />
</center>

Can we use them in neural MT?

### Attention mechanism

Jointly learning to align and to translate.

<center>
    <img src="mt_figures/encdec_att.svg" width="20%" />
</center>

### Attention matrix

<center>
    <img src="mt_figures/att_matrix.png" width="40%" />
</center>

<div style="text-align: right;">
    (from <a href="https://arxiv.org/abs/1409.0473">Bahdanau et al., 2014</a>)
</div>

### Transformers

Replace LSTMs by self-attention. Attend to encoded input *and* to partial output (autoregressive).

<center>
    <img src="http://jalammar.github.io/images/xlnet/transformer-encoder-decoder.png" width=50%/>
</center>

<div style="text-align: right;">
    (from <a href="http://jalammar.github.io/illustrated-gpt2/">The Illustrated GPT-2</a>)
</div>

# MT evaluation

We're training the model with *negative log-likelihood*, but that's not the best way to *evaluate* it.

**Danish source:** Forskerne analyserede resultaterne.

Possible English translations:

* The researchers analysed the results.
* The researchers conducted an analysis of the results.
* The scientists examined the findings.

Different word forms and wording can express the same event. We will compare these with translations that change **number** or **negation**.

## Human evaluation

* **Accuracy** (meaning preservation) evaluates the translation model.
* **Fluency** evaluates the target-language output.
* The Multidimensional Quality Metrics framework uses an explicit error taxonomy for more diagnostic evaluation.

See [Freitag et al., 2021](https://aclanthology.org/2021.tacl-1.87/).

In general, manual evaluation is the best way, but it is not scalable. **Automatic** metrics are therefore often used.

## BLEU score

A *reference-based* metric ([Papineni et al., 2002](https://aclanthology.org/P02-1040/)):

* Compute **modified $n$-gram precision** for $n=1,\ldots,4$: clip each candidate count to its maximum count in any reference, then divide by the candidate's total $n$-gram count.
* Take the **geometric mean** of these precisions.
* Apply a **brevity penalty** if the output is shorter than the reference length.

$$\mathrm{BLEU}=\mathrm{BP}\cdot\exp\left(\frac{1}{4}\sum_{n=1}^{4}\log p_n\right)$$

The definition uses **0–1**; SacreBLEU reports **0–100**. A high score measures overlap, not necessarily preserved meaning.

Recommended library: [SacreBLEU](https://github.com/mjpost/sacrebleu).

The examples below use smoothed **sentence BLEU**. For system evaluation, compute **corpus BLEU** from pooled counts; do not average sentence BLEU scores.

### One example, three metrics

**Source:** Forskerne analyserede resultaterne.

**Reference:** The researchers analysed the results.

Compare an exact match, two faithful rewordings, a **number error** (`researchers` → `researcher`), and a **negation error**. We will use the same five candidates for BLEU, chrF and BERTScore.

In [ ]:
%pip install -q sacrebleu

In [ ]:
import pandas as pd
from IPython.display import HTML
from sacrebleu.metrics import BLEU

mt_reference = "The researchers analysed the results."
evaluation_examples = pd.DataFrame([
    ("Exact match", mt_reference),
    ("Rewording", "The researchers conducted an analysis of the results."),
    ("Paraphrase", "The scientists examined the findings."),
    ("Number error", "The researcher analysed the results."),
    ("Negation error", "The researchers did not analyse the results."),
], columns=["Case", "Translation"])

def show_evaluation(metric_columns, translations=True):
    columns = ["Case"] + (["Translation"] if translations else []) + metric_columns
    table = evaluation_examples[columns].to_html(
        index=False, classes="mt-evaluation",
        formatters={"BLEU": "{:.1f}".format, "chrF": "{:.1f}".format,
                    "BERTScore F1": "{:.3f}".format})
    return HTML("""<style>
        table.mt-evaluation {width:100%; border-collapse:collapse;}
        table.mt-evaluation th, table.mt-evaluation td {
            font-size:22px !important; line-height:1.3;
            padding:0.35em; text-align:left !important;
            white-space:normal; vertical-align:top;
        }
        </style>""" + "<p><b>Reference:</b> " + mt_reference + "</p>" + table)

bleu = BLEU(effective_order=True)
evaluation_examples["BLEU"] = [
    bleu.sentence_score(candidate, [mt_reference]).score
    for candidate in evaluation_examples["Translation"]
]
show_evaluation(["BLEU"])

Case,Translation,BLEU
Exact match,The researchers analysed the results.,100.0
Rewording,The researchers conducted an analysis of the results.,22.3
Paraphrase,The scientists examined the findings.,10.7
Number error,The researcher analysed the results.,53.7
Negation error,The researchers did not analyse the results.,25.8


The faithful rewording and paraphrase score below the **number error**. Word overlap does not reliably rank meaning preservation.

`analysed` → `analysis`: a related word form no longer matches as a whole word, and the rewording also breaks longer word $n$-grams.

Negation reverses the statement, yet much of the wording still matches. Check the source when interpreting a metric score.

## Beyond BLEU

Word $n$-gram overlap can miss:

* **Related word forms:** `analysed` / `analysis` share a stem but are different word tokens.
* **Synonyms:** `researchers` / `scientists`, `results` / `findings`.
* **Meaning-changing edits:** a plural ending or a negation can change the message while leaving most words intact.

Character overlap can give partial credit for shared word forms. Embedding similarity can help with paraphrases. Neither replaces checking meaning against the source.

### chrF

**chrF** measures **character $n$-gram** F-score between hypothesis and reference ([Popović, 2015](https://aclanthology.org/W15-3049/)).

$$\mathrm{chrF}_\beta=(1+\beta^2)\frac{\mathrm{chrP}\cdot\mathrm{chrR}}{\beta^2\mathrm{chrP}+\mathrm{chrR}}$$

* chrP and chrR average precision and recall over character $n$-gram orders.
* Shared stems receive partial credit despite different word endings or token boundaries.
* No language-specific word tokenizer is needed. SacreBLEU defaults to $\beta=2$ and reports **0–100**.

**chrF++** also includes word $n$-grams (`word_order=2` in SacreBLEU).

In [ ]:
from sacrebleu.metrics import CHRF

chrf = CHRF()
evaluation_examples["chrF"] = [
    chrf.sentence_score(candidate, [mt_reference]).score
    for candidate in evaluation_examples["Translation"]
]
show_evaluation(["BLEU", "chrF"])

Case,Translation,BLEU,chrF
Exact match,The researchers analysed the results.,100.0,100.0
Rewording,The researchers conducted an analysis of the results.,22.3,71.8
Paraphrase,The scientists examined the findings.,10.7,18.6
Number error,The researcher analysed the results.,53.7,88.8
Negation error,The researchers did not analyse the results.,25.8,77.6


`researchers` → `researcher` changes only one character. chrF gives substantial partial credit, although the **number of people has changed**.

`analysed` / `analysis` retain character overlap in the faithful rewording. The lexical paraphrase receives much less credit because its wording is different.

Partial credit is useful for word-form variation. It cannot tell us whether an inflection change is acceptable or a factual error, or whether negation reverses the meaning.

### BERTScore: contextual similarity

BERTScore ([Zhang et al., 2020](https://arxiv.org/abs/1904.09675)) matches contextual token embeddings to compute precision, recall and F1.

* It can recognize similarities that word/character overlap misses.
* Test it on the **same five candidates**, including the number and negation errors.
* We report **baseline-rescaled F1**, not a percentage; its scale differs from BLEU and chrF and it can be negative.

In [ ]:
%pip install -q bert-score

In [ ]:
import os
from bert_score import BERTScorer

# Use the restricted checkpoint loader even with older Transformers versions.
os.environ["TORCH_FORCE_WEIGHTS_ONLY_LOAD"] = "1"

# Use the published checkpoint without requesting a background conversion.
os.environ["DISABLE_SAFETENSORS_CONVERSION"] = "1"

scorer = BERTScorer(model_type="microsoft/deberta-large-mnli", lang="en", rescale_with_baseline=True)

# Avoid an undefined tokenizer limit overflowing in Transformers 5.
scorer._tokenizer.model_max_length = scorer._model.config.max_position_embeddings

P, R, F1 = scorer.score(
    evaluation_examples["Translation"].tolist(),
    [mt_reference] * len(evaluation_examples),
)
evaluation_examples["BERTScore F1"] = F1.cpu().tolist()
show_evaluation(["BLEU", "chrF", "BERTScore F1"], translations=False)

Case,BLEU,chrF,BERTScore F1
Exact match,100.0,100.0,1.000
Rewording,22.3,71.8,0.773
Paraphrase,10.7,18.6,0.798
Number error,53.7,88.8,0.957
Negation error,25.8,77.6,0.574


Compare the **faithful rewording and paraphrase** with the **number and negation errors**. Does each metric's ranking agree with your judgement?

BERTScore ranks the faithful rewordings above the negation error, but the **number error still outranks both faithful alternatives**. Check these meaning-changing edits against the source.

These are sentence-level demonstrations. To compare MT systems, evaluate a held-out corpus and inspect errors alongside the aggregate scores.

## Summary: MT evaluation

* Human evaluation is the best, but doing it well is expensive
* Automatic metrics are sometimes a good proxy:
    * BLEU
    * ChrF
    * BERTScore

This is also applicable to other text generation tasks, e.g., multilingual QA!

## Improving efficiency and quality in MT

+ More data
+ Bigger models
+ Better neural network architectures
+ Semi-supervised learning
+ **Transfer learning**

## Further reading

* [Jurafsky & Martin, Chapter 13: Machine Translation](https://web.stanford.edu/~jurafsky/slp3/13.pdf)
* Non-neural machine translation:
  * Ilya Pestov, [A history of machine translation from the Cold War to deep learning](https://www.freecodecamp.org/news/a-history-of-machine-translation-from-the-cold-war-to-deep-learning-f1d335ce8b5/)
  * Mike Collins, [Lecture notes on IBM Models 1 and 2](https://www.cs.columbia.edu/~mcollins/courses/nlp2011/notes/ibm12.pdf)
* Sequence-to-sequence models:
  * Graham Neubig, [Neural Machine Translation and Sequence-to-sequence Models: A Tutorial](https://arxiv.org/abs/1703.01619)
* Further refinements and challenges:
  * Philipp Koehn, [Neural Machine Translation, §13.6–13.8](https://arxiv.org/abs/1709.07809)